In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
from imblearn.over_sampling import SMOTE

In [35]:
df = pd.read_csv("../data/processed/deliveries_features_ready.csv")

In [36]:
df.shape

(43648, 66)

In [42]:
df.columns

Index(['Agent_Age', 'Agent_Rating', 'Category', 'Pickup_Next_Day', 'Distance',
       'Extreme_Distance_Flag', 'Night_Order_Flag', 'Order_Hour',
       'Weekend_Flag', 'Peak_Hour_Flag', 'Weather_Fog', 'Weather_Sandstorms',
       'Weather_Stormy', 'Weather_Sunny', 'Weather_Windy', 'Traffic_Jam',
       'Traffic_Low', 'Traffic_Medium', 'Vehicle_scooter', 'Vehicle_van',
       'Area_Other', 'Area_Urban', 'Distance_Bucket_5–10',
       'Distance_Bucket_10–20', 'Distance_Bucket_20–50',
       'Distance_Bucket_50–100', 'Distance_Bucket_100–300',
       'Distance_Bucket_300+', 'Traffic_Weather_High_Fog',
       'Traffic_Weather_High_Sandstorms', 'Traffic_Weather_High_Stormy',
       'Traffic_Weather_High_Sunny', 'Traffic_Weather_High_Windy',
       'Traffic_Weather_Jam_Cloudy', 'Traffic_Weather_Jam_Fog',
       'Traffic_Weather_Jam_Sandstorms', 'Traffic_Weather_Jam_Stormy',
       'Traffic_Weather_Jam_Sunny', 'Traffic_Weather_Jam_Windy',
       'Traffic_Weather_Low_Cloudy', 'Traffic_Weather_

In [43]:
df["Category"].value_counts(normalize=True)

Category
On-Time    0.501787
At Risk    0.298914
Delayed    0.199299
Name: proportion, dtype: float64

In [44]:
X = df.drop(columns=["Category"])
y = df["Category"]

In [45]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [46]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
label_mapping

{'At Risk': np.int64(0), 'Delayed': np.int64(1), 'On-Time': np.int64(2)}

On-Time  → 2
At Risk  → 0
Delayed  → 1

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
np.bincount(y_train)

array([10438,  6959, 17521])

In [50]:
np.bincount(y_test)

array([2609, 1740, 4381])

In [52]:
numeric_cols = [
    "Agent_Age",
    "Agent_Rating",
    "Distance",
    "Order_Hour"
]
numeric_cols

['Agent_Age', 'Agent_Rating', 'Distance', 'Order_Hour']

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [57]:
pd.Series(y_train).value_counts(normalize=True)

2    0.501776
0    0.298929
1    0.199295
Name: proportion, dtype: float64

In [58]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))
class_weight_dict

{0: np.float64(1.1150922909880565),
 1: np.float64(1.6725583177659626),
 2: np.float64(0.6643075927934098)}

In [59]:
X_train_scaled.shape

(34918, 64)

In [60]:
X_test_scaled.shape

(8730, 64)

In [61]:
X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)

pd.Series(y_train).to_csv("../data/processed/y_train.csv", index=False)
pd.Series(y_test).to_csv("../data/processed/y_test.csv", index=False)